# Session 2 notebook: factor models for a fund, with the iShares Momentum ETF

In Session 1 we used one stock, Micron. Here we use one **fund**, the iShares MSCI USA Momentum Factor ETF (MTUM), and ask what it is exposed to. Exercise 2 then asks you to do the same for Vanguard's Value ETF (VTV).

| Part | Question | Lecture link |
|---|---|---|
| A | Get MTUM's monthly returns and the factors, in the same units | Session 2 notebook |
| B | How much of MTUM is the market? (CAPM) | FW Chapter 1, Jensen regression |
| C | Add size and value. What does Fama-French three-factor say? | Fama-French (1993) |
| D | Add momentum. What changes, and why? | Carhart (1997), Jacobs and Levy |
| E | The same regression by matrix algebra | Matrix math for factor models |
| F | Your turn: Exercise 2 on VTV | Exercise 2 |

Run the cells from top to bottom. Each part ends with **How to read the output**.

Data: Yahoo Finance (MTUM prices) and Ken French's data library (factors). This is for learning, not trading advice.

> **A note on how this notebook was made.** It was written by an AI (Claude, from Anthropic), working from your instructor's lecture plan. Treat it the way you would treat any code you did not write: run it, read it, and check its claims against the output you see. If something looks wrong, tell your instructor.

# Part A. Data: MTUM and the factors, in the same units

1. Download MTUM's daily prices and keep the last close of each month.
2. Compute monthly returns **in percent**, because Ken French's factors are in percent per month.
3. Download the Fama-French three factors and the momentum factor.
4. Line everything up by month and compute MTUM's excess return (return minus the risk-free rate).

MTUM started trading in April 2013, so we have about 13 years of monthly data.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import pandas_datareader.data as web

# 1. MTUM daily prices (adjusted for dividends), through August 2026
ticker = 'MTUM'
prices = yf.download(ticker, start='2013-01-01', end='2026-09-01', auto_adjust=True, progress=False)
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.get_level_values(0)

# 2. Monthly returns in percent: last close of each month, then percent change
monthly = prices['Close'].resample('ME').last().pct_change().dropna() * 100
monthly.index = monthly.index.to_period('M')   # monthly periods, to match Ken French's dates
fund = monthly.rename('Fund')

# 3. Fama-French three factors and momentum (both already in percent per month)
ff = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='2013-01-01')[0]
mom = web.DataReader('F-F_Momentum_Factor', 'famafrench', start='2013-01-01')[0]
mom.columns = [c.strip() for c in mom.columns]   # the column is named 'Mom   ' with spaces

# 4. Merge on month and compute the excess return
df = pd.concat([fund, ff, mom], axis=1, join='inner').dropna()
df['Excess'] = df['Fund'] - df['RF']
print(len(df), "months:", df.index[0], "to", df.index[-1])
df.tail()

**How to read the output.** You should see about 160 months, from mid-2013 to August 2026, and columns for the fund's return, `Mkt-RF`, `SMB`, `HML`, `RF`, `Mom` and `Excess`, all in percent per month.

**Watch the units.** If you forget the `* 100`, the fund's returns are decimals (0.012) while the factors are percent (1.2). The regressions still run, but alpha and the loadings come out wrong by a factor of 100. Always check a few rows by eye.

# Part B. The CAPM: how much of MTUM is the market?

The Jensen regression from Session 1, now for a fund:

**Excess = alpha + beta x Mkt-RF + e**

In [ ]:
capm = sm.OLS(df['Excess'], sm.add_constant(df[['Mkt-RF']])).fit()
print(capm.summary().tables[1])
print(f"\nR-squared = {capm.rsquared:.2f}")

**How to read the output (our run, May 2013 to August 2026)**

| Measure | Value | Reading |
|---|---|---|
| alpha (`const`) | 0.15% per month, t = 0.7 | Not significant |
| beta (`Mkt-RF`) | 0.95, t = 20.4 | Moves about one-for-one with the market |
| R² | 0.73 | Low for a diversified fund |

Lecture rule: a diversified long-only fund usually has an R² **above 90%**. MTUM holds a diversified portfolio of large and mid-cap U.S. stocks, so 0.73 is a warning sign. Either the data are wrong, or the model is missing something MTUM is exposed to.

# Part C. Fama-French three-factor: add size and value

**Excess = alpha + beta x Mkt-RF + s x SMB + h x HML + e**

In [ ]:
ff3 = sm.OLS(df['Excess'], sm.add_constant(df[['Mkt-RF', 'SMB', 'HML']])).fit()
print(ff3.summary().tables[1])
print(f"\nR-squared = {ff3.rsquared:.2f}")

**How to read the output (our run)**

| Loading | Value | Reading |
|---|---|---|
| `SMB` | -0.14, t = -1.9 | A slight large-cap tilt, not significant at 5% |
| `HML` | -0.18, t = -3.2 | Significant and negative: this model says MTUM is a **growth** fund |
| R² | 0.75 | Barely better than the CAPM, still well below 90% |

Should we believe "MTUM is a growth fund"? Not yet. A momentum fund buys whatever went up last year. If growth stocks led the market, MTUM holds growth stocks, and HML picks that up because momentum is not in the model. Part D tests that.

# Part D. Carhart four-factor: add momentum

**Excess = alpha + beta x Mkt-RF + s x SMB + h x HML + u x Mom + e**

In [ ]:
carhart = sm.OLS(df['Excess'], sm.add_constant(df[['Mkt-RF', 'SMB', 'HML', 'Mom']])).fit()
print(carhart.summary().tables[1])
print(f"\nR-squared = {carhart.rsquared:.2f}")

# Side by side: how each loading changes as factors are added
compare = pd.DataFrame({'CAPM': capm.params, 'FF3': ff3.params, 'Carhart': carhart.params})
compare = compare.reindex(['const', 'Mkt-RF', 'SMB', 'HML', 'Mom'])
compare.loc['R-squared'] = [capm.rsquared, ff3.rsquared, carhart.rsquared]
compare.round(2)

**How to read the output (our run)**

| | CAPM | FF3 | Carhart |
|---|---|---|---|
| beta (`Mkt-RF`) | 0.95 | 0.98 | 1.09 |
| `HML` | | -0.18 | -0.02 |
| `Mom` | | | **0.49**, t = 12.9 |
| R² | 0.73 | 0.75 | **0.88** |

- **Momentum is what MTUM is.** A loading of 0.49 is large (FW: even 0.2 or 0.3 is substantial), and R² jumps from 0.75 to 0.88.
- **The "growth" loading disappears.** Once momentum is in the model, HML is essentially zero. In Part C, HML was standing in for the missing momentum factor. This is the Jacobs and Levy point: a "naive" loading can be a proxy for something else.
- **Beta goes up.** Over this period the momentum factor was negatively correlated with the market, so leaving it out pulled the market beta down too. Adding a factor can change every other loading.
- **Low R² for a fund means a data error *or* a missing factor.** Here it was a missing factor.
- **Alpha** stays insignificant in all three models.

# Part E. The same regression by matrix algebra

Every regression in this notebook is one formula from the lecture: **b = (X'X)<sup>-1</sup> X'y**, where X has a column of ones (for alpha) and one column per factor. Let's compute it by hand and check it against statsmodels.

The same loadings also split the fund's variance into a factor part and an idiosyncratic part: **Var(y) = b' Omega b + Var(e)**, where Omega is the covariance matrix of the factors. This is the one-fund version of **Sigma = B Omega B' + D**.

In [ ]:
factors = ['Mkt-RF', 'SMB', 'HML', 'Mom']
y = df['Excess'].values
X = np.column_stack([np.ones(len(y)), df[factors].values])   # column of ones for alpha

b = np.linalg.inv(X.T @ X) @ X.T @ y
print("by hand:     ", np.round(b, 4))
print("statsmodels: ", np.round(carhart.params.values, 4))

# Variance split: factor part b' Omega b, plus residual variance
betas = b[1:]
omega = np.cov(df[factors].values, rowvar=False)
resid = y - X @ b
factor_var = betas @ omega @ betas
resid_var = np.var(resid, ddof=1)
print(f"\nVar(y) = {np.var(y, ddof=1):.2f}   factor part = {factor_var:.2f}   residual part = {resid_var:.2f}")
print(f"Factor share of variance = {factor_var / np.var(y, ddof=1):.2f}  (compare with R-squared = {carhart.rsquared:.2f})")

**How to read the output.** The two rows of coefficients should match to every decimal shown: statsmodels is doing exactly this matrix calculation. The factor share of variance equals R², because R² *is* the share of the fund's variance explained by its factor exposures.

# Part F. Exercise 2 (due Oct 26, noon)

These are the same five tasks as on the Week 1 page:

1. Retrieve Vanguard's Value ETF (VTV) monthly returns.
2. Retrieve Fama-French monthly factor data.
3. Merge the two datasets.
4. Regress the VTV return on Mkt-RF. What is the alpha, and what is the factor loading on market risk?
5. Regress the VTV return on all the Fama-French factors. What is the alpha, and what are the factor loadings on market risk and value (HML)?

**Tips (not extra tasks):**

- Parts A to D above are the template. Change the ticker to `'VTV'` and the start date to `'2004-01-01'` (VTV started trading in 2004).
- "Regress the VTV return" means its excess return, VTV minus `RF`, as in this notebook. Keep both sides in percent.
- To interpret the HML loading, see the Session 2 slide "How big should a value fund's HML loading be?"
